# Test Bloods Notebook

This notebook tests bloods data from the `basic_observations` Elasticsearch index. Validates data extraction through the pat2vec pipeline and merged output.

In [ ]:
import os
import random
import shutil
import sys

import numpy as np

In [ ]:
random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
TEMP_DIR = "/tmp/test_bloods_test_project"
PROJ_NAME = TEMP_DIR
DB_FILENAME = "temp_bloods_db.sqlite"
DB_PATH = os.path.join(TEMP_DIR, "outputs", DB_FILENAME)
print(f"Using temp directory: {TEMP_DIR}")

In [ ]:
try:
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR, ignore_errors=True)
        print(f"Cleaned up: {TEMP_DIR}")
except Exception as e:
    msg = f"Failed to clean up directory {TEMP_DIR}: {e}."
    raise RuntimeError(msg)

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

print("Starting Elasticsearch container...")
if not es_container.start():
    msg = "Failed to start Elasticsearch."
    raise RuntimeError(msg)

host, username, password = es_container.get_credentials()

creds_filename = "test_elastic_credentials_bloods_get.py"
creds_content = f"""\nusername = \"{username}\"
password = \"{password}\"
api_key = None
hosts = ["{host}"]
"""

with open(creds_filename, "w") as f:
    f.write(creds_content)

print(f"Created {creds_filename}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

grandparent_dir = "/workspaces/pat2vec"
schema_path = os.path.join(grandparent_dir, "test_files", "elastic_schemas.json")

config_populate = config_class(
    proj_name=TEMP_DIR,
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
    test_data_path="/workspaces/pat2vec/notebooks/test_files/treatment_docs.csv",
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print(f"Population complete. Generated {len(patient_ids)} dummy patients.")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
import time

time.sleep(2)
print("Indices refreshed.")

In [ ]:
DB_FILENAME = "temp_bloods_db.sqlite"
DB_PATH = os.path.join(TEMP_DIR, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    msg = f"Failed to remove database: {e}."
    raise RuntimeError(msg)

db_connection_string = "sqlite:///" + DB_PATH
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.main_pat2vec import main

print("main imported successfully")

import pandas as pd

from pat2vec.util.post_processing_build_methods import build_merged_bloods

In [ ]:
config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={
        "bloods": True,
        "demo": False,
        "bmi": False,
        "drugs": False,
        "diagnostics": False,
        "core_02": False,
        "bed": False,
        "vte_status": False,
        "hosp_site": False,
        "core_resus": False,
        "news": False,
        "smoking": False,
        "appointments": False,
        "covid": False,
        "epic_encounters": False,
        "epic_clinical_notes": False,
        "epic_medical_history": False,
        "epic_orders": False,
        "epic_lab_results": False,
        "epic_patients": False,
        "epic_imaging_reports": False,
        "epic_clinical_notes_appointments": False,
        "annotations": False,
        "annotations_mrc": False,
        "textual_obs": False,
        "annotations_reports": False,
        "negated_presence_annotations": False,
        "epic_clinical_notes_annotations": False,
        "epic_medical_history_annotations": False,
        "epic_imaging_reports_annotations": False,
        "epic_orders_annotations": False,
        "epic_clinical_notes_appointments_annotations": False,
    },
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    check_patient_existence=False,
    db_connection_string=db_connection_string,
    treatment_doc_filename="test_files/treatment_docs.csv",
)
pat2vec_obj = main(
    cogstack=True,
    use_filter=False,
    json_filter_path=None,
    random_seed_val=random_seed_value,
    hostname=None,
    config_obj=config_obj,
)

In [ ]:
print("\n=== PROCESSING PATIENTS WITH pat_maker ===")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

try:
    print(f"Processing patient 0: {pat2vec_obj.all_patient_list[0]}")
    pat2vec_obj.pat_maker(0)
except Exception as e:
    msg = (
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    )
    raise RuntimeError(
        msg,
    ) from e

print("Patient feature extraction complete.")

In [ ]:
# Verify bloods data was processed and saved
print("\n=== VERIFYING BLOODS DATA ===")

from pat2vec.util.helper_functions import get_df_from_db

# Check raw_bloods table exists and has data
try:
    bloods_df = get_df_from_db(
        config_obj,
        "raw_data",
        "raw_bloods",
        patient_ids=pat2vec_obj.all_patient_list,
    )
    print(f"Retrieved {len(bloods_df)} bloods records from database")

    if bloods_df.empty:
        msg = (
            "FATAL ERROR: No bloods data found in raw_bloods table. "
            "The bloods processing pipeline failed."
        )
        raise RuntimeError(
            msg,
        )

    print(f"Bloods column names: {list(bloods_df.columns)}")
except Exception as e:
    msg = f"Failed to verify bloods data: {e}"
    raise RuntimeError(msg) from e

In [ ]:
def merge_bloods_data(config_obj, patient_list):
    """Merge bloods data and raise ValueError if empty or file doesn't exist."""
    merged_path = build_merged_bloods(patient_list, config_obj, overwrite=True)

    if not os.path.exists(merged_path):
        msg = "MERGE FAILED: Merged file not found"
        raise ValueError(msg)

    merged_df = pd.read_csv(merged_path)

    if len(merged_df) == 0:
        msg = "MERGE FAILED: Merged file is empty"
        raise ValueError(msg)

    return merged_df

In [ ]:
try:
    merged_df = merge_bloods_data(config_obj, pat2vec_obj.all_patient_list)
    print(f"Merged bloods data: {len(merged_df)} rows")
except ValueError as e:
    msg = (
        f"merge_bloods_data() raised ValueError: {e}. "
        "Critical error - no data found for bloods."
    )
    raise RuntimeError(
        msg,
    ) from e

In [ ]:
merged_batches_dir = os.path.join(TEMP_DIR, "merged_batches")
try:
    if os.path.exists(merged_batches_dir):
        shutil.rmtree(merged_batches_dir, ignore_errors=True)
        print(f"Cleaned merged batches: {merged_batches_dir}")
except Exception as e:
    msg = f"Failed to clean merged batches: {e}."
    raise RuntimeError(msg)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    msg = f"Failed to remove database file '{DB_PATH}': {e}. Critical error - cleanup incomplete."
    raise RuntimeError(
        msg,
    ) from e

try:
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR, ignore_errors=False)
        print(f"Removed project directory: {TEMP_DIR}")
except Exception as e:
    msg = f"Failed to remove '{TEMP_DIR}' directory: {e}. Critical error - cleanup incomplete."
    raise RuntimeError(
        msg,
    ) from e

try:
    if os.path.exists(creds_filename):
        os.remove(creds_filename)
        print(f"Removed Elasticsearch credentials: {creds_filename}")
except Exception as e:
    msg = (
        f"Failed to remove Elasticsearch credentials file '{creds_filename}': {e}. "
        "Critical error - cleanup incomplete."
    )
    raise RuntimeError(
        msg,
    ) from e

In [ ]:
print("\n=== FINAL VERIFICATION ===")

assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(TEMP_DIR), "Project directory still exists!"
assert not os.path.exists(
    creds_filename,
), "Elasticsearch credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")